# Genera el footage de LA JUNTA DE LOS HORÓMETROS (gratis, GPU T4 de Colab)

**RUTA**: Archivo > Subir cuaderno > elige `GENERA_FOOTAGE_COLAB.ipynb`... o cópialo. Luego menú **Entorno de ejecución / Runtime > Cambiar tipo** → `T4 GPU`.
Ejecuta las celdas en orden (▶▶▶ todo). Genera las 7 REF, luego los 45 clips (2 variantes c/u) y baja un ZIP.

**Datos honestos**: modelo open-source (LTX-Video), calidad buena pero inferior a Veo/Kling y la consistencia facial depende de la imagen REF. T4 gratis puede tardar ~2-4 min por clip; si la sesión muere, re-ejecuta las celdas (reanuda donde quedó).

**Flujo**: corre TODO una vez con PRUEBA=True (solo toma T001) para revisar calidad, y luego pon PRUEBA=False en esa celda y vuelve a ejecutarla para las 45 tomas. Los clips salen de ~6.4 s (161 fotogramas a 25 fps) e irán a 16:9 final en CapCut.

Al terminar: descomprime y copia `refs/` y `clips/` a la carpeta `footage/` del proyecto y corre `python pipeline/gen.py assemble`.

In [ ]:
!pip -q install -U "diffusers>=0.32.0" "transformers>=4.45.0" "accelerate>=1.0.0" "sentencepiece" "protobuf<6.0.0" imageio imageio-ffmpeg safetensors pillow peft

In [ ]:
import os, json, csv, io, zipfile, gc, time
import torch
from PIL import Image
from diffusers import FluxPipeline, LTXImageToVideoPipeline
from diffusers.utils import export_to_video
from tqdm.auto import tqdm

DUR_FRAMES = 161   # ~6.4 s a 25 fps (los clips saldran ~6.4 s, no la duracion exacta del take)
W, H = 704, 512
FPS = 25
NUM_STEPS = 50
VARIANTES = 2
SEED = 4242
DTYPE = torch.bfloat16   # calidad; si muy lento en T4, probar torch.float16
NEG = ("worst quality, inconsistent motion, blurry, jittery, distorted, texto, letras, "
       "subtitulos, marca de agua, logotipo")
FOOT = "/content/footage"
REFS = os.path.join(FOOT, "refs")
CLIPS = os.path.join(FOOT, "clips")
os.makedirs(REFS, exist_ok=True)
os.makedirs(CLIPS, exist_ok=True)
print("entorno listo")


In [ ]:
JOBS_CSV = r"""take,escena,dur_s,ref,ref_tipo,prompt,negative
1,T001,8,LOC-EXTERIOR,lugar,"Vista aérea cinematográfica de una mina mexicana llamada La Providencia, caminos de terracería, instalaciones industriales, maquinaria pesada trabajando, camiones y camionetas circulando, tarde soleada, ambiente de operación minera activa, cámara avanzando lentamente hacia las instalaciones., Dron aereo avanzando lentamente hacia las instalaciones, estabilidad cinematografica, tarde soleada. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
2,T002,6,LOC-EXTERIOR,lugar,"Scooptram minero saliendo lentamente de una zona de trabajo, polvo ligero en el ambiente, trabajadores con casco caminando a distancia, escena industrial minera realista, cámara lateral suave., Camara lateral suave siguiendo al scooptram, ligero desenfoque de parallax al fondo. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
3,T003,7,LOC-EXTERIOR,lugar,"Trabajadores de mina caminando hacia las oficinas al final de una jornada parcial, algunos conversando, ambiente tranquilo de lunes por la tarde, maquinaria trabajando al fondo., Steadicam siguiendo de frente a los trabajadores, fondo desenfocado, maquinaria en movimiento. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
4,T004,6,LOC-OFICINA,lugar,"Reloj de oficina marcando exactamente las tres de la tarde, acercamiento cinematográfico lento, ambiente silencioso, sensación de que algo importante está por comenzar., Acercamiento macro lento al reloj, camara fija, enfoque suave, tension creciente. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
5,T005,6,REF-01,persona,"Ing. Molina entrando a una oficina minera sencilla pero profesional, llevando una carpeta y una laptop, casco blanco bajo el brazo, preparándose para una reunión importante., Pan lateral suave siguiendo al ingeniero entrar a la oficina, carpeta y laptop a la mano. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
6,T006,7,REF-01,persona,"Ing. Molina sentado frente a una computadora portátil, conectándose a una videollamada de trabajo, pantalla mostrando varias personas en una reunión virtual genérica, ambiente profesional., Camara fija frontal, encuadre medio; el ingeniero se sienta y abre la laptop para conectarse. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
7,T007,6,REF-01,persona,"Primer plano del ing. Molina ajustando el micrófono y revisando documentos, expresión tranquila y concentrada., Primer plano con poca profundidad de campo, camara fija enfocando manos y microfono. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
8,T008,7,REF-01,persona,"Ing. Molina presentando tranquilamente diapositivas con gráficas de producción minera en una pantalla grande, actitud segura y profesional., Panoramica lenta sobre las diapositivas y despues empuje lento hacia el ingeniero. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
9,T009,6,REF-02,persona,"Pantalla de presentación empresarial mostrando gráficas genéricas, producción, avances y proyectos mineros, sin texto legible específico., Pantalla de presentacion, camara casi fija con respiracion minima, graficas sin texto legible. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
10,T010,5,REF-01,persona,"Ing. Molina respondiendo preguntas en una videollamada, asintiendo con confianza., Camara fija con zoom-in muy lento (105 a 112 por ciento) mientras el ingeniero responde con confianza. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
11,T011,7,REF-03,persona,"Varios trabajadores de mantenimiento sentados en una oficina cercana escuchando la junta, relajados, algunos tomando café., Plano general fijo, grupo relajado con cafe, voz de fondo de la junta. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
12,T012,7,LOC-OFICINA,lugar,"Pantalla de computadora mostrando dashboard moderno de mantenimiento con indicadores, gráficas, disponibilidad de equipos, mantenimiento preventivo y lecturas de maquinaria., Empuje lento hacia el dashboard de mantenimiento, respiracion minima de camara. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
13,T013,6,REF-01,persona,"Primer plano del ing. Molina observando una pantalla con varios indicadores, expresión tranquila., Camara fija sobre el hombro, enfoque en la pantalla con indicadores de maquinaria. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
14,T014,6,REF-02,persona,"Ingeniero senior llamado Maldonado apareciendo serio en una videollamada empresarial, mirando directamente a cámara como si estuviera a punto de hacer una pregunta importante., Camara fija frontal mirando directo a camara, zoom-in gradual que aumenta la tension. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
15,T015,7,REF-01,persona,"Primer plano cinematográfico del ing. Molina mirando fijamente una pantalla de computadora, expresión confundida y preocupada, permanece inmóvil durante varios segundos., Camara fija, plano medio; el ingeniero se queda completamente inmóvil, solo el parpadeo. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
16,T016,5,REF-01,persona,"Ing. Molina mirando lentamente hacia un lado buscando apoyo de sus compañeros de mantenimiento, nadie responde., Pan lateral muy lento (unos 15 grados) mientras el ingeniero busca apoyo, nadie responde. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
17,T017,6,REF-03,persona,"Grupo de trabajadores de mantenimiento sentados completamente serios, evitando contacto visual, uno mira al piso, otro mira hacia otro lado, humor visual sutil., Camara fija en grupo; los rostros evitan la camara: uno al piso, otro a un lado, timing comico. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
18,T018,5,LOC-OFICINA,lugar,"Primer plano de una gráfica de maquinaria donde varios valores aparecen vacíos o en cero, estilo dashboard empresarial., Macro sobre la grafica con valores en cero, camara fija, silencio. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
19,T019,6,REF-03,persona,"Tres mecánicos de mantenimiento sentados uno al lado del otro en una oficina minera, completamente serios, intentando no mirar al ing. Molina, escena cómica estilo falso documental., Camara fija falso documental; tres mecanicos completamente inmóviles, miradas fijas al frente. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
20,T020,5,REF-03,persona,"Uno de los mecánicos mira discretamente hacia otro trabajador y después vuelve inmediatamente la mirada hacia el frente., Camara fija; movimiento rapido de ojos y mirada discreta del mecanico hacia un lado y regreso. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
21,T021,6,REF-01,persona,"Ing. Molina gira lentamente la cabeza hacia mantenimiento esperando ayuda, pero todos permanecen inmóviles., Camara fija; giro de cabeza muy lento del ingeniero hacia mantenimiento, todos inmóviles. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
22,T022,5,REF-01,persona,"Molina sentado frente a computadora durante videollamada, intentando responder varias preguntas rápidamente, ligeramente nervioso pero manteniendo actitud profesional., Camara fija; ritmo rapido de miradas entre pantalla, hojas y camara, nerviosismo contenido. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
23,T023,5,REF-02,persona,"Maldonado en videollamada haciendo preguntas serias y revisando documentos., Camara fija frontal sobre Maldonado, zoom-in lento de presion, documentos a la mano. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
24,T024,5,REF-03,persona,"Grupo de mantenimiento mirando entre ellos discretamente mientras escuchan preguntas difíciles., Camara fija; miradas discretas e intercambios rapidos de ojos entre los mecanicos. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
25,T025,7,REF-01,persona,"Ing. Molina revisando rápidamente hojas impresas, computadora y teléfono mientras participa en una videollamada, expresión seria y ligeramente estresada., Handheld ligero que crece en nerviosismo, paginas pasando rapido, telefono y pantalla. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
26,T026,5,LOC-OFICINA,lugar,"Primer plano de hojas de mantenimiento con tablas de maquinaria, números y registros técnicos, mano pasando páginas rápidamente., Macro de hojas de mantenimiento y manos; camara fija, paginas pasando con prisa. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
27,T027,5,REF-01,persona,"Molina asentando repetidamente mientras escucha instrucciones por videollamada., Camara fija; asentir repetido con cadencia casi comica mientras escucha instrucciones. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
28,T028,5,LOC-OFICINA,lugar,"Reloj de oficina avanzando, ambiente de cansancio después de una reunión larga., Acercamiento lento al reloj, cansancio despues de la junta larga. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
29,T029,6,REF-01,persona,"Ing. Molina frente a laptop escuchando las últimas palabras de una videollamada, expresión cansada., Camara fija; fatiga en el rostro, casi sin movimiento, luz de pantalla. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
30,T030,4,LOC-OFICINA,lugar,"Primer plano de una mano terminando una videollamada en una computadora., Macro sobre la mano cerrando la videollamada; camara fija, silencio al terminar. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
31,T031,6,REF-01,persona,"Ing. Molina cerrando lentamente una laptop después de una reunión complicada, expresión seria, oficina minera silenciosa., Camara fija; el ingeniero cierra la laptop muy lentamente, expresion seria, silencio. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
32,T032,6,REF-01,persona,"Ing. Molina se levanta lentamente y mira hacia varios trabajadores de mantenimiento, todos permanecen en silencio., Pan vertical suave al incorporarse y luego plano general de la oficina en silencio. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
33,T033,6,REF-03,persona,"Trabajador de mantenimiento intenta saludar amigablemente al ing. Molina pero el ingeniero continúa caminando serio sin responder, comedia visual ligera., Camara fija; el trabajador saluda amigable y el ingeniero pasa de largo, comedia por reaccion. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
34,T034,6,REF-03,persona,"Tres trabajadores de mantenimiento afuera de una oficina minera conversando discretamente y tratando de contener la risa., Handheld discreto falso documental; tres trabajadores conteniendo la risa afuera. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
35,T035,6,REF-03,persona,"Trabajador contando una historia divertida a sus compañeros mientras imita la expresión seria de un ingeniero., Handheld; un trabajador imita el gesto serio del ingeniero y los demas rien contenido. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
36,T036,5,LOC-EXTERIOR,lugar,"Dos trabajadores conversando cerca del almacén de una mina mientras revisan materiales., Handheld documental; conversacion junto al almacen mientras revisan materiales. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
37,T037,5,LOC-EXTERIOR,lugar,"Mecánicos conversando junto a maquinaria pesada en un taller minero., Handheld; conversacion natural junto a maquinaria pesada en el taller. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
38,T038,5,LOC-EXTERIOR,lugar,"Trabajadores conversando junto a una camioneta de mina y riéndose discretamente., Handheld; risa discreta junto a una camioneta de mina. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
39,T039,6,LOC-EXTERIOR,lugar,"Grupo pequeño de trabajadores mineros riéndose mientras uno cuenta una historia con muchos gestos., Handheld; uno cuenta la historia con gestos grandes y el grupo se rie. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
40,T040,7,LOC-EXTERIOR,lugar,"Martes por la mañana en taller minero, varios trabajadores revisando maquinaria con tablets y hojas de inspección, todos trabajando rápidamente., Plano general con empuje lento; varios trabajadores activos con tablets y hojas. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
41,T041,6,LOC-EXTERIOR,lugar,"Técnico revisando el horómetro digital de una máquina minera y anotando la lectura en una tablet., Primer plano nitido; el tecnico lee el horometro digital y anota en la tablet. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
42,T042,6,LOC-EXTERIOR,lugar,"Mecánico tomando lectura de horómetro de una máquina diésel mientras otro trabajador registra información., Plano medio; leer el horometro diesel mientras otro trabajador registra. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
43,T043,8,LOC-EXTERIOR,lugar,"Vista panorámica cinematográfica de una mina al atardecer, maquinaria pesada trabajando a distancia, cielo naranja, ambiente épico pero tranquilo., Dron lento panoramico al atardecer, cielo naranja, maquinaria a distancia, tono epico. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
44,T044,6,REF-01,persona,"Ing. Molina caminando por instalaciones de la mina al atardecer, casco blanco, actitud seria pero tranquila., Steadicam siguiendo al ingeniero caminando al atardecer, actitud seria pero tranquila. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing"
45,T045,6,LOC-EXTERIOR,lugar,"Equipo de mantenimiento trabajando correctamente y registrando lecturas de maquinaria en computadoras y tablets, ambiente organizado., Plano general ordenado; el equipo registra lecturas en tablets y computadoras. Estilo: cinematografico realista, mina subterranea y oficinas mineras en Mexico, trabajadores con casco, chaleco reflejante y uniforme de trabajo, ambiente profesional pero con comedia ligera, iluminacion natural, camara cinematografica, personajes consistentes entre escenas, formato horizontal 16:9, sin texto generado dentro de la imagen.","texto, letras, palabras, subtitulos, marca de agua, logotipo, sobreimpresion, manos deformes, dedos extra, rostro inconsistente, cambio de identidad del personaje, luz parpadeante, saltos de fotograma, morphing""""

REFS_PLAN = {
 "REF-01": "Hombre mexicano de aproximadamente 45-50 anos, ingeniero de mina, casco blanco, camisa de trabajo, chaleco reflejante, aspecto profesional, serio pero expresivo, retrato de medio cuerpo hacia la camara, fondo de oficina minera desenfocado.",
 "REF-02": "Hombre mexicano de aproximadamente 50 anos, ejecutivo o ingeniero senior, aspecto serio, camisa formal, personalidad firme, retrato frontal de videollamada, fondo de oficina ejecutiva.",
 "REF-03": "Hombre mexicano de aproximadamente 30 anos, mecanico de mina, casco amarillo, overol, chaleco reflejante, retrato de medio cuerpo, fondo de taller desenfocado.",
 "REF-04": "Hombre mexicano de aproximadamente 40 anos, mecanico de mina, casco amarillo, overol gastado, bigote, retrato de medio cuerpo, fondo de taller desenfocado.",
 "REF-05": "Hombre mexicano de aproximadamente 25 anos, mecanico de mina, casco amarillo, camisa de trabajo azul, retrato de medio cuerpo, fondo de taller desenfocado.",
 "LOC-OFICINA": "Oficina minera sencilla pero profesional de un ingeniero en Mexico: escritorio metalico, laptop abierta, pizarra con tablas, ventana con luz natural, casco blanco colgado, ambiente de mina, videollamada en pantalla.",
 "LOC-EXTERIOR": "Mina mexicana La Providencia: patio industrial, camionetas, maquinaria pesada, casas de oficinas, cerros al fondo, tarde soleada, ambiente de operacion minera activa."
}

jobs = list(csv.DictReader(io.StringIO(JOBS_CSV)))
for j in jobs: j["dur_s"] = int(j["dur_s"])
print("jobs:", len(jobs), "| refs:", len(REFS_PLAN))

In [ ]:
print("Cargando FLUX.1-schnell (REFs)...")
img_pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=DTYPE)
img_pipe.enable_model_cpu_offload()

print("Cargando LTX-Video (I2V)...")
vid_pipe = LTXImageToVideoPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=DTYPE)
vid_pipe.enable_model_cpu_offload()   # encoders en CPU, transformer en GPU (seguro para T4 16GB)
try:
    vid_pipe.vae.enable_vae_slicing()
except Exception:
    pass
try:
    vid_pipe.vae.enable_tiling()
except Exception:
    pass
print("modelos listos")


In [ ]:
print("Generando las 7 imagenes REF (FLUX.1-schnell)...")
absent = [r for r in REFS_PLAN if not os.path.exists(os.path.join(REFS, r + ".png"))]
for r in tqdm(absent, desc="REFs"):
    img = img_pipe(prompt=REFS_PLAN[r] + ", formato panoramico 16:9, alta calidad",
                   guidance_scale=0.0, num_inference_steps=4,
                   width=1024, height=576,
                   generator=torch.Generator().manual_seed(SEED)).images[0]
    img.save(os.path.join(REFS, r + ".png"))
print("REFs listas:", len(REFS_PLAN) - len(absent), "de", len(REFS_PLAN))

del img_pipe           # libera memoria de FLUX para LTX
gc.collect()
torch.cuda.empty_cache()
print("memoria liberada")


In [ ]:
PRUEBA = True   # si True genera SOLO la toma T001 (prueba de calidad); pon False para las 45
state = {}
sf = os.path.join(FOOT, "state.json")
if os.path.exists(sf):
    state = json.load(open(sf))

DESDE, HASTA = 1, 45   # edita para reanudar rangos tras una desconexion
tgt = [x for x in jobs if DESDE <= int(x["take"]) <= HASTA]
if PRUEBA:
    tgt = [x for x in tgt if int(x["take"]) == 1]
    print("MODO PRUEBA: solo T001. Revisa clips/T001_v1.mp4 y pon PRUEBA=False.")

for j in tqdm(tgt):
    esc = j["escena"]
    dv = state.get(esc, [])
    for n in range(len(dv), max(VARIANTES, len(dv))):
        out = os.path.join(CLIPS, "%s_v%d.mp4" % (esc, n + 1))
        if os.path.exists(out):
            dv.append(out); continue
        ref_path = os.path.join(REFS, j["ref"] + ".png") if j["ref"] else None
        image = Image.open(ref_path).resize((W, H)) if ref_path else None
        kw = dict(
            prompt=j["prompt"] + ". Escena en vivo cinematografica, camara estable, realismo fotografico.",
            negative_prompt=NEG + ", " + j["negative"],
            width=W, height=H, num_frames=DUR_FRAMES, frame_rate=FPS,
            num_inference_steps=NUM_STEPS, guidance_scale=3.0,
            decode_timestep=0.05, decode_noise_scale=0.025,
            max_sequence_length=512,
            generator=torch.Generator().manual_seed(SEED + int(j["take"]) * 11 + n))
        if image is not None:
            kw["image"] = image
        t0 = time.time()
        try:
            frames = vid_pipe(**kw).frames[0]
            export_to_video(frames, out, fps=FPS)
            dv.append(out)
            state[esc] = dv
            json.dump(state, open(sf, "w"))
            torch.cuda.empty_cache()
            print("OK %s_v%d  (%.1f min)" % (esc, n + 1, (time.time() - t0) / 60))
        except Exception as e:
            print("ERROR", esc, e)
hechos = sum(len(state.get(x, [])) for x in state)
print("clips generados:", hechos, "/", len(jobs) * VARIANTES)


In [ ]:
zp = "/content/footage.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(FOOT):
        for f in files:
            p = os.path.join(root, f)
            z.write(p, os.path.relpath(p, "/content"))
print("zip:", zp, os.path.getsize(zp) // 1024, "KB")
try:
    from google.colab import files
    files.download(zp)
except Exception:
    print("Descarga manual:", zp)
